In [ ]:
import pandas as pd
from functools import reduce
pd.set_option('display.max_columns', None)


# Path to your Excel file
file_path = 'orange msaied1.xlsx'  # <-- change to your actual file name

# Load all sheets into a dictionary
dfs = pd.read_excel(file_path, sheet_name=None)

# Confirm sheet names
print("Loaded sheets:", dfs.keys())

# Rename columns in each DataFrame to include year suffix
yearly_dfs = []
for year, df in dfs.items():
    year_str = str(year)
    df = df.copy()
    df = df.drop(columns=['year'], errors='ignore')  # drop 'year' column if it exists
    df['name'] = df['name'].str.lower()
    df['name'] = df['name'].str.replace("orange one - ",'')
    df['name'] = df['name'].str.replace("orange one  - ",'')

    df = df.rename(columns={
        'units': f'units_{year_str}',
        'total': f'total_{year_str}',
        'rate': f'rate_{year_str}'
    })
    yearly_dfs.append(df)

# Merge all years on 'name'
df_merged = reduce(lambda left, right: pd.merge(left, right, on='name', how='outer'), yearly_dfs)

# Sort columns: name first, then the rest sorted
cols = ['name'] + sorted([col for col in df_merged.columns if col != 'name'])
df_merged = df_merged[cols]

df_merged = df_merged.fillna(0)

# Calculate differences and percentage changes
years = sorted([str(year) for year in dfs.keys()])

for i in range(1, len(years)):
    prev_year = years[i-1]
    curr_year = years[i]
    
    # Absolute differences
    df_merged[f'units_diff_{curr_year}_vs_{prev_year}'] = (
        df_merged[f'units_{curr_year}'] - df_merged[f'units_{prev_year}']
    )
    df_merged[f'total_diff_{curr_year}_vs_{prev_year}'] = (
        df_merged[f'total_{curr_year}'] - df_merged[f'total_{prev_year}']
    )
    df_merged[f'rate_diff_{curr_year}_vs_{prev_year}'] = (
        df_merged[f'rate_{curr_year}'] - df_merged[f'rate_{prev_year}']
    )

    print(df_merged[f'rate_{curr_year}'])
    print( df_merged[f'rate_{prev_year}'])
    print(df_merged[f'rate_diff_{curr_year}_vs_{prev_year}'] )
    
    
    # Optional: percentage change
    # df_merged[f'units_pct_change_{curr_year}_vs_{prev_year}'] = (
    #     (df_merged[f'units_{curr_year}'] - df_merged[f'units_{prev_year}']) / df_merged[f'units_{prev_year}']
    # ) * 100
    
    # df_merged[f'total_pct_change_{curr_year}_vs_{prev_year}'] = (
    #     (df_merged[f'total_{curr_year}'] - df_merged[f'total_{prev_year}']) / df_merged[f'total_{prev_year}']
    # ) * 100
    
    # df_merged[f'rate_pct_change_{curr_year}_vs_{prev_year}'] = (
    #     (df_merged[f'rate_{curr_year}'] - df_merged[f'rate_{prev_year}']) / df_merged[f'rate_{prev_year}']
    # ) * 100
# Fill NaN with 0 if you prefer (optional)
df_merged = df_merged.fillna(0)

# Save to Excel (for Power BI)
output_file = 'hotels_comparison.xlsx'
df_merged.to_excel(output_file, index=False)


print(f"✅ Merged data prepared and saved to '{output_file}' successfully!")


In [52]:
file = pd.ExcelFile("orange msaied1.xlsx")
year2020 = pd.read_excel(file,sheet_name="2020")
year2021 = pd.read_excel(file,sheet_name="2021")
year2022 = pd.read_excel(file,sheet_name="2022")
year2023 = pd.read_excel(file,sheet_name="2023")
year2024 = pd.read_excel(file,sheet_name="2024")

In [53]:
all = pd.concat([year2020,year2021,year2022,year2023,year2024])
all['name'] = all['name'].str.lower()
all['name'] = all['name'].str.replace("orange one - ",'')
all['name'] = all['name'].str.replace("orange one  - ",'')

In [54]:
all.to_csv('orange_units_revenue_year.csv',index=False)